In [1]:
import pandas as pd
import glob  # Biblioteca para encontrar arquivos
import os
import sys

print("Script 'processar_lote.py' iniciado.")

# --- 1. CONSTANTES (Copiadas do seu Notebook) ---
COL_VEICULO = 'COD'
COL_LINHA = 'CODIGOLINHA'
COL_SENTIDO = 'SENT'
COL_TIMESTAMP_API = 'request_timestamp'
COL_LAT = 'LAT'
COL_LON = 'LON'
CASE_ID = 'case:concept:name'

COLS_DUPLICATAS = [
    CASE_ID, 'REFRESH', COL_LAT, COL_LON, 
    COL_LINHA, 'SITUACAO', COL_SENTIDO
]

# --- 2. FUNÇÕES DE LIMPEZA (Copiadas da Célula 2) ---
# (Removi os 'prints' de dentro das funções para o log não ficar poluído)

def clean_dataframe(df_raw):
    """Limpa o DataFrame bruto, converte tipos e ordena os dados."""
    df = df_raw.copy()
    try:
        df[COL_TIMESTAMP_API] = pd.to_datetime(df[COL_TIMESTAMP_API])
    except Exception as e:
        print(f"  -> Erro ao converter timestamp: {e}. Pulando etapa de ordenação.")
        return df 

    # Ordena por veículo e depois por tempo
    df = df.sort_values(by=[COL_VEICULO, COL_TIMESTAMP_API])
    
    # Preenche dados ausentes (ffill = forward fill)
    if COL_LINHA in df.columns:
        df[COL_LINHA] = df[COL_LINHA].ffill()
    if COL_SENTIDO in df.columns:
        df[COL_SENTIDO] = df[COL_SENTIDO].ffill()
        
    return df

def create_case_id(df):
    """Cria um 'Case ID' único para cada viagem, quebrando por veículo, linha ou sentido."""
    if df.empty or COL_LINHA not in df.columns or COL_SENTIDO not in df.columns:
        print("  -> Aviso: Colunas 'CODIGOLINHA' ou 'SENT' ausentes. Não foi possível criar Case ID.")
        df[CASE_ID] = "default_case"
        return df

    # Identificar as "quebras"
    mudou_veiculo = df[COL_VEICULO] != df[COL_VEICULO].shift(1)
    mudou_linha = df[COL_LINHA] != df[COL_LINHA].shift(1)
    mudou_sentido = df[COL_SENTIDO] != df[COL_SENTIDO].shift(1)
    
    quebra_de_viagem = mudou_veiculo | mudou_linha | mudou_sentido
    id_grupo_viagem = quebra_de_viagem.cumsum()
    
    df[CASE_ID] = df[COL_VEICULO].astype(str) + '_' + id_grupo_viagem.astype(str)
    return df

# --- 3. FUNÇÃO DE PROCESSAMENTO POR DIA ---

def processar_um_dia(data_alvo_str, pasta_raiz_brutos, pasta_saida):
    """
    Processa todos os arquivos de um único dia e salva um CSV limpo para esse dia.
    data_alvo_str (str): 'YYYY-MM-DD'
    """
    print(f"\n--- Processando Dia: {data_alvo_str} ---")
    
    # 1. Encontrar todos os arquivos *horários* para este dia
    #    Caminho: ../Coleta_Planejamento/dados_em_tempo_real/2025-11-02/2025-11-02_*.csv
    padrao_busca_dia = os.path.join(pasta_raiz_brutos, data_alvo_str, f"{data_alvo_str}_*.csv")
    arquivos_horarios = glob.glob(padrao_busca_dia)
    
    if not arquivos_horarios:
        print(f"  -> Aviso: Nenhum arquivo CSV encontrado para o dia {data_alvo_str} no padrão '{padrao_busca_dia}'")
        return

    print(f"  Encontrados {len(arquivos_horarios)} arquivos horários. Combinando...")
    
    # 2. Combinar todos os arquivos horários
    lista_dfs_brutos = []
    for f in arquivos_horarios:
        try:
            lista_dfs_brutos.append(pd.read_csv(f))
        except Exception as e:
            print(f"    -> Erro ao ler {f}: {e}")
    
    if not lista_dfs_brutos:
        print(f"  -> Erro: Nenhum arquivo horário pôde ser lido para {data_alvo_str}.")
        return

    df_dia_bruto = pd.concat(lista_dfs_brutos, ignore_index=True)
    print(f"  -> Total de {len(df_dia_bruto)} linhas brutas para {data_alvo_str}.")

    # 3. Aplicar o Pipeline de Limpeza (Lógica da Célula 4)
    print("  Iniciando pipeline de limpeza...")
    
    df_clean = clean_dataframe(df_dia_bruto) 
    df_com_cases_bruto = create_case_id(df_clean)
    
    # Filtro 1: Situação NaN (REC, garagem, etc.)
    df_filtrado_1 = df_com_cases_bruto[df_com_cases_bruto['SITUACAO'].notna()]
    
    # Filtro 2: "Glitches" da API
    df_filtrado_2 = df_filtrado_1[df_filtrado_1['SITUACAO'] != 'NÃO CONFORMIDADE']
    
    # Filtro 3: Pings estacionários (dados repetidos)
    df_limpo_final = df_filtrado_2.drop_duplicates(subset=COLS_DUPLICATAS, keep='first').copy()
    
    print(f"  -> Limpeza concluída. {len(df_limpo_final)} linhas válidas restantes.")

    # 4. Salvar o arquivo limpo DESTE DIA
    nome_arquivo_saida = f"{data_alvo_str}_limpo.csv"
    caminho_saida = os.path.join(pasta_saida, nome_arquivo_saida)
    
    df_limpo_final.to_csv(caminho_saida, index=False, encoding='utf-8-sig')
    print(f"  -> Sucesso! Arquivo limpo salvo em: {caminho_saida}")


# --- 4. EXECUÇÃO DO SCRIPT ---
if __name__ == "__main__":
    
    # --- CONFIGURAÇÃO ---
    # 1. Esta é a lista de dias que você quer processar
    #    (Datas da "semana passada" que você baixou da UFPR)
    DATAS_ALVO = [
        "2025-11-02",
        "2025-11-03",
        "2025-11-04",
        "2025-11-05",
        "2025-11-06",
        "2025-11-07",
        "2025-11-08"
    ]
    
    # 2. Pasta raiz dos seus dados brutos (relativo ao local do script)
    #    Script está em: TCC_Process_Mining/
    #    Dados estão em: Coleta_Planejamento/dados_em_tempo_real/
    PASTA_RAIZ_DADOS_BRUTOS = '../Coleta_Planejamento/dados_em_tempo_real'
    
    # 3. Pasta onde salvar os resultados (relativo ao local do script)
    PASTA_DADOS_LIMPOS = '../Dados_TCC/dados_limpos_por_dia'
    # --------------------

    os.makedirs(PASTA_DADOS_LIMPOS, exist_ok=True)
    
    if not os.path.isdir(PASTA_RAIZ_DADOS_BRUTOS):
        print(f"ERRO: A pasta de entrada '{PASTA_RAIZ_DADOS_BRUTOS}' não foi encontrada.")
        print("Verifique o caminho relativo do script.")
        sys.exit()

    print(f"Iniciando processamento em lote para {len(DATAS_ALVO)} dias...")
    
    # Loop principal que chama a função para cada dia
    for dia in DATAS_ALVO:
        processar_um_dia(dia, PASTA_RAIZ_DADOS_BRUTOS, PASTA_DADOS_LIMPOS)

    print("\n--- Processamento de todos os dias alvo concluído! ---")

Script 'processar_lote.py' iniciado.
Iniciando processamento em lote para 7 dias...

--- Processando Dia: 2025-11-02 ---
  Encontrados 24 arquivos horários. Combinando...
  -> Total de 1012029 linhas brutas para 2025-11-02.
  Iniciando pipeline de limpeza...
  -> Limpeza concluída. 126248 linhas válidas restantes.
  -> Sucesso! Arquivo limpo salvo em: ../Dados_TCC/dados_limpos_por_dia/2025-11-02_limpo.csv

--- Processando Dia: 2025-11-03 ---
  Encontrados 24 arquivos horários. Combinando...
  -> Total de 991708 linhas brutas para 2025-11-03.
  Iniciando pipeline de limpeza...
  -> Limpeza concluída. 269460 linhas válidas restantes.
  -> Sucesso! Arquivo limpo salvo em: ../Dados_TCC/dados_limpos_por_dia/2025-11-03_limpo.csv

--- Processando Dia: 2025-11-04 ---
  Encontrados 24 arquivos horários. Combinando...
  -> Total de 984115 linhas brutas para 2025-11-04.
  Iniciando pipeline de limpeza...
  -> Limpeza concluída. 268648 linhas válidas restantes.
  -> Sucesso! Arquivo limpo salvo em